# 02. Scaling Laws

## 학습 목표
- LLM 성능이 모델 크기, 데이터, 컴퓨팅에 어떻게 의존하는지 이해
- Kaplan Scaling Laws의 Power Law 관계 시각화
- Chinchilla 논문의 Compute-Optimal Training 인사이트 파악
- 제한된 예산으로 어떤 모델을 학습할지 판단하는 방법 이해

## 핵심 논문
- [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) (Kaplan et al., 2020)
- [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556) (Chinchilla, Hoffmann et al., 2022)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

## 1. Scaling Laws 개요: 모델 성능은 무엇에 의존하는가

LLM의 성능(Loss)은 세 가지 요소에 의해 결정된다:

| 요소 | 기호 | 설명 |
|------|------|------|
| Parameters | $N$ | 모델의 파라미터 수 (모델 크기) |
| Data | $D$ | 학습 데이터의 토큰 수 |
| Compute | $C$ | 학습에 사용된 컴퓨팅 (FLOPs) |

**핵심 발견**: 성능은 이 세 요소에 대해 **Power Law(\uba71법칙)** 을 따른다.

$$L(x) = \left(\frac{x_0}{x}\right)^{\alpha}$$

즉, log-log 그래프에서 **직선**으로 나타난다.

**떠오르는 질문**: 같은 컴퓨팅 예산으로 큰 모델을 적게 학습할까, 작은 모델을 많이 학습할까?

In [ ]:
# Power Law의 직관적 이해
# y = (x0 / x)^alpha: x가 커질수록 y는 감소하지만 점점 느려진다

x = np.logspace(0, 4, 100)  # 1 ~ 10000

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 선형 스케일
ax = axes[0]
for alpha in [0.05, 0.1, 0.2]:
    y = (1.0 / x) ** alpha + 1.5  # baseline loss 추가
    ax.plot(x, y, label=f'alpha={alpha}', linewidth=2)
ax.set_xlabel('Scale (N, D, or C)')
ax.set_ylabel('Loss')
ax.set_title('Power Law (Linear Scale)')
ax.legend()
ax.grid(True, alpha=0.3)

# Log-Log 스케일 (직선으로 보임)
ax = axes[1]
for alpha in [0.05, 0.1, 0.2]:
    y = (1.0 / x) ** alpha
    ax.loglog(x, y, label=f'alpha={alpha}', linewidth=2)
ax.set_xlabel('Scale (log)')
ax.set_ylabel('Loss (log)')
ax.set_title('Power Law (Log-Log Scale) -> Straight Line')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\u2192 log-log plot에서 직선 = Power Law 관계")
print("\u2192 alpha가 클수록 scale up의 효과가 큼")

---
## 2. Kaplan Scaling Laws (2020)

OpenAI의 Kaplan et al.이 발견한 세 가지 Power Law:

### Loss vs Parameters (N)
$$L(N) = \left(\frac{N_0}{N}\right)^{\alpha_N}, \quad \alpha_N \approx 0.076$$

### Loss vs Data (D)
$$L(D) = \left(\frac{D_0}{D}\right)^{\alpha_D}, \quad \alpha_D \approx 0.095$$

### Loss vs Compute (C)
$$L(C) = \left(\frac{C_0}{C}\right)^{\alpha_C}, \quad \alpha_C \approx 0.050$$

**Kaplan의 결론**: 컴퓨팅 예산이 제한될 때, **모델을 키우는 것**이 데이터를 늘리는 것보다 효과적이다.

In [ ]:
# Kaplan Scaling Laws 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Kaplan et al. 논문의 실제 수치에 가까운 값
# Loss vs Parameters
N = np.logspace(6, 10, 50)  # 1M ~ 10B parameters
N0 = 8.8e13
alpha_N = 0.076
L_N = (N0 / N) ** alpha_N

ax = axes[0]
ax.loglog(N, L_N, 'b-', linewidth=2)
ax.set_xlabel('Parameters (N)')
ax.set_ylabel('Test Loss')
ax.set_title('Loss vs Parameters\nalpha_N = 0.076')
ax.grid(True, alpha=0.3, which='both')

# 주요 모델 표시
model_params = {'GPT-2 Small': 124e6, 'GPT-2 Large': 774e6, 'GPT-3': 175e9}
# GPT-3는 범위 밖이므로 표시하지 않음

# Loss vs Data
D = np.logspace(8, 12, 50)  # 100M ~ 1T tokens
D0 = 5.4e13
alpha_D = 0.095
L_D = (D0 / D) ** alpha_D

ax = axes[1]
ax.loglog(D, L_D, 'r-', linewidth=2)
ax.set_xlabel('Dataset Size (D, tokens)')
ax.set_ylabel('Test Loss')
ax.set_title('Loss vs Data\nalpha_D = 0.095')
ax.grid(True, alpha=0.3, which='both')

# Loss vs Compute
C = np.logspace(15, 24, 50)  # FLOPs
C0 = 3.1e8
alpha_C = 0.050
L_C = (C0 / C) ** alpha_C

ax = axes[2]
ax.loglog(C, L_C, 'g-', linewidth=2)
ax.set_xlabel('Compute (C, FLOPs)')
ax.set_ylabel('Test Loss')
ax.set_title('Loss vs Compute\nalpha_C = 0.050')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("alpha_D(0.095) > alpha_N(0.076) > alpha_C(0.050)")
print("-> 데이터 10배 늘리는 것이 모델 10배 키우는 것보다 Loss 감소 효과가 큼")
print("   하지만 Kaplan은 '모델 크기 우선'으로 해석 -> 나중에 Chinchilla가 반박")

---
## 3. Chinchilla 논문: Compute-Optimal Training (2022)

DeepMind의 Hoffmann et al.이 Kaplan의 결론을 **반박**했다.

### Kaplan vs Chinchilla

| | Kaplan (2020) | Chinchilla (2022) |
|---|---|---|
| 최적 전략 | 모델을 키우고 데이터는 적당히 | 모델과 데이터를 **균형적**으로 |
| 토큰/파람 비율 | ~5:1 | **~20:1** |
| 핵심 | 큰 모델이 유리 | 적절한 크기 + 충분한 데이터 |

### Chinchilla의 법칙

$$D_{\text{optimal}} \approx 20 \times N$$

- 파라미터 10B인 모델 -> 200B 토큰으로 학습해야 최적
- Gopher(280B 파람, 300B 토큰)는 **데이터 부족** (1:1 비율)
- Chinchilla(70B 파람, 1.4T 토큰)이 Gopher보다 성능 우수 (20:1 비율)

### Compute-Optimal 배분

Compute $C \approx 6ND$ (forward + backward pass)

$C$가 주어졌을 때:
$$N_{\text{opt}} \propto C^{0.5}, \quad D_{\text{opt}} \propto C^{0.5}$$

-> 컴퓨팅을 2배 늘리면, 모델 크기와 데이터를 **둘 다** $\sqrt{2}$배 늘려야 한다.

In [ ]:
# Chinchilla Optimal: 토큰 수 = 20 x 파라미터 수
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 다양한 모델의 토큰/파람 비율
ax = axes[0]

models = {
    'GPT-3':      {'N': 175e9,  'D': 300e9,  'color': 'royalblue'},
    'Gopher':     {'N': 280e9,  'D': 300e9,  'color': 'coral'},
    'Chinchilla': {'N': 70e9,   'D': 1400e9, 'color': 'green'},
    'LLaMA-7B':   {'N': 7e9,    'D': 1000e9, 'color': 'purple'},
    'LLaMA-65B':  {'N': 65e9,   'D': 1400e9, 'color': 'darkviolet'},
}

for name, info in models.items():
    ratio = info['D'] / info['N']
    ax.scatter(info['N'] / 1e9, info['D'] / 1e9, s=150, color=info['color'],
               edgecolors='black', zorder=5)
    ax.annotate(f"{name}\n(ratio={ratio:.0f}:1)",
                (info['N'] / 1e9, info['D'] / 1e9),
                textcoords='offset points', xytext=(10, 5), fontsize=9)

# Chinchilla optimal line: D = 20N
N_line = np.linspace(1, 300, 100)
ax.plot(N_line, 20 * N_line, 'k--', alpha=0.5, label='Chinchilla Optimal (D=20N)')

ax.set_xlabel('Parameters (Billions)')
ax.set_ylabel('Training Tokens (Billions)')
ax.set_title('Model Size vs Training Data')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 310)
ax.set_ylim(0, 6500)

# 오른쪽: Compute-Optimal 배분
ax = axes[1]

C_budget = np.logspace(18, 25, 100)  # FLOPs

# Kaplan: 모델 우선
N_kaplan = (C_budget / 6) ** 0.73  # N에 더 많이 배분
D_kaplan = C_budget / (6 * N_kaplan)

# Chinchilla: 균형 배분
N_chinchilla = (C_budget / (6 * 20)) ** 0.5
D_chinchilla = 20 * N_chinchilla

ax.loglog(C_budget, N_kaplan, 'b--', label='N (Kaplan)', linewidth=2)
ax.loglog(C_budget, D_kaplan, 'b:', label='D (Kaplan)', linewidth=2)
ax.loglog(C_budget, N_chinchilla, 'r-', label='N (Chinchilla)', linewidth=2)
ax.loglog(C_budget, D_chinchilla, 'r-', label='D (Chinchilla)', linewidth=2, alpha=0.5)

ax.set_xlabel('Compute Budget (FLOPs)')
ax.set_ylabel('Optimal N or D')
ax.set_title('Compute Budget -> Optimal N and D')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("Kaplan: 컴퓨팅 증가 -> 대부분을 모델 키우는 데 사용")
print("Chinchilla: 컴퓨팅 증가 -> 모델과 데이터를 동등하게 키움")

---
## 4. 실제 LLM 학습 설정 비교

실제 모델들이 Chinchilla 법칙을 얼마나 따르는지 확인해보자.

In [ ]:
# 실제 LLM 학습 설정 비교 테이블
llm_data = [
    # (Model, Parameters(B), Tokens(B), Compute(FLOPs), Year, Chinchilla Ratio)
    ('GPT-3',         175,    300,    '3.1e23',  2020),
    ('Gopher',        280,    300,    '5.0e23',  2021),
    ('Chinchilla',     70,   1400,    '5.0e23',  2022),
    ('PaLM',          540,    780,    '2.5e24',  2022),
    ('LLaMA-7B',        7,   1000,    '4.2e22',  2023),
    ('LLaMA-65B',      65,   1400,    '5.5e23',  2023),
    ('LLaMA-2-70B',    70,   2000,    '8.4e23',  2023),
    ('Mistral-7B',      7,   8000,    '3.4e23',  2023),
]

print(f"{'Model':<18} {'Params(B)':>10} {'Tokens(B)':>10} {'FLOPs':>12} {'D/N Ratio':>10} {'Chinchilla?':>12}")
print("-" * 80)

for name, params, tokens, flops, year in llm_data:
    ratio = tokens / params
    optimal = 'Optimal' if 15 <= ratio <= 25 else ('Under-trained' if ratio < 15 else 'Over-trained')
    status_marker = '  ' if 15 <= ratio <= 25 else ('<-' if ratio < 15 else '>>')
    print(f"{name:<18} {params:>10.0f} {tokens:>10.0f} {flops:>12} {ratio:>9.1f}x {optimal:>12}")

print()
print("\ud574\uc11d:")
print("  - GPT-3, Gopher: D/N < 5 -> Chinchilla 기준 심각한 Under-training")
print("  - Chinchilla: D/N = 20 -> 정확히 최적")
print("  - LLaMA: D/N >> 20 -> Chinchilla보다 더 많은 데이터로 학습 (inference 비용 최적화)")
print("  - Mistral-7B: D/N ~ 1143 -> 작은 모델을 아주 많은 데이터로 학습 (over-training)")

In [ ]:
# 모델별 D/N 비율 시각화
fig, ax = plt.subplots(figsize=(10, 6))

names = [d[0] for d in llm_data]
ratios = [d[2] / d[1] for d in llm_data]
years = [d[4] for d in llm_data]

# Chinchilla optimal 범위
ax.axhspan(15, 25, alpha=0.15, color='green', label='Chinchilla Optimal (15-25x)')
ax.axhline(y=20, color='green', linestyle='--', alpha=0.5)

colors = ['royalblue' if r < 15 else ('green' if r <= 25 else 'coral') for r in ratios]
bars = ax.bar(names, ratios, color=colors, edgecolor='black', alpha=0.8)

# 바 위에 비율 표시
for bar, ratio in zip(bars, ratios):
    display_ratio = f'{ratio:.0f}x' if ratio < 100 else f'{ratio:.0f}x'
    y_pos = min(bar.get_height() + 5, 200)
    ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
            display_ratio, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Tokens / Parameters Ratio (D/N)')
ax.set_title('Tokens-to-Parameters Ratio by Model')
ax.set_ylim(0, max(ratios) * 1.15)
ax.legend()

plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 5. Scaling Laws 시각화: Log-Log Plot으로 Power Law 확인

Kaplan 논문의 발견을 재현해보자.

실제 논문의 데이터를 기반으로, 모델 크기를 늘릴 때 Loss가 어떻게 변하는지 시뮬레이션한다.

In [ ]:
# 논문 Figure를 재현하는 시뮬레이션

# Kaplan의 공식: L(N) = (N_c / N)^alpha + L_inf
# alpha ~ 0.076, L_inf ~ 1.69 (irreducible loss)
N_c = 8.8e13
alpha = 0.076
L_inf = 1.69  # Entropy of natural text (줄일 수 없는 손실)

N_range = np.logspace(5, 11, 200)  # 100K ~ 100B parameters
L_pred = (N_c / N_range) ** alpha + L_inf

# 학습 데이터가 충분할 때의 Loss (가상 실험 결과)
np.random.seed(42)
N_points = np.array([1e5, 5e5, 1e6, 5e6, 1e7, 5e7, 1e8, 5e8, 1e9, 5e9, 1e10])
L_points = (N_c / N_points) ** alpha + L_inf
# 약간의 노이즈 추가
noise = np.random.normal(0, 0.01, len(N_points))
L_points_noisy = L_points + noise

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Loss vs Parameters
ax = axes[0]
ax.loglog(N_range, L_pred - L_inf, 'b-', linewidth=2, label='Power Law fit')
ax.scatter(N_points, L_points_noisy - L_inf, c='red', s=50, zorder=5, label='Simulated experiments')
ax.set_xlabel('Non-Embedding Parameters (N)')
ax.set_ylabel('Test Loss - L_inf (reducible loss)')
ax.set_title('Kaplan: Loss vs Parameters (log-log)')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# 오른쪽: 세 요소 동시 비교
ax = axes[1]

# Loss vs N
x_norm = np.logspace(0, 5, 100)
L_n = x_norm ** (-0.076)
L_d = x_norm ** (-0.095)
L_c = x_norm ** (-0.050)

ax.loglog(x_norm, L_n, 'b-', linewidth=2, label=f'vs Parameters (alpha={0.076})')
ax.loglog(x_norm, L_d, 'r-', linewidth=2, label=f'vs Data (alpha={0.095})')
ax.loglog(x_norm, L_c, 'g-', linewidth=2, label=f'vs Compute (alpha={0.050})')

ax.set_xlabel('Scale (normalized)')
ax.set_ylabel('Reducible Loss')
ax.set_title('Three Power Laws Compared')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\uc67c\ucabd: log-log에\uc11c 직\uc120 -> Power Law \ud655\uc778")
print("\uc624\ub978\ucabd: alpha\uac00 \ud074\uc218\ub85d \uae30\uc6b8\uae30\uac00 \uac00\ud30c\ub974\ub2e4 -> scale up \ud6a8\uacfc\uac00 \ud06c\ub2e4")
print("  -> \ub370\uc774\ud130(alpha=0.095)\ub97c \ub298\ub9ac\ub294 \uac83\uc774 \ucef4\ud4e8\ud305(alpha=0.050)\uc744 \ub298\ub9ac\ub294 \uac83\ubcf4\ub2e4 \ud6a8\uacfc\uc801")

---
## 6. 실용적 의미: 제한된 예산으로 어떤 모델을 학습할 것인가

### 컴퓨팅 예산 계산

- 학습 FLOPs $\approx 6ND$ (forward 2ND + backward 4ND)
- A100 GPU: ~312 TFLOPS (BF16), 실제 효율 ~50% -> ~156 TFLOPS
- GPU-hours = FLOPs / (TFLOPS * 3600 * 1e12)

In [ ]:
# Compute Budget Calculator

def compute_training_cost(N_params, D_tokens, gpu_tflops=156, gpu_cost_per_hour=2.0):
    """
    학습 비용 계산
    N_params: 파라미터 수
    D_tokens: 토큰 수
    gpu_tflops: GPU의 실효 TFLOPS
    gpu_cost_per_hour: GPU 시간당 비용 (USD)
    """
    flops = 6 * N_params * D_tokens
    gpu_seconds = flops / (gpu_tflops * 1e12)
    gpu_hours = gpu_seconds / 3600
    cost = gpu_hours * gpu_cost_per_hour
    return flops, gpu_hours, cost


def chinchilla_optimal(compute_budget_flops):
    """주어진 Compute 예산에서 Chinchilla optimal N, D 계산"""
    # C = 6 * N * D, D = 20 * N -> C = 6 * N * 20 * N = 120 * N^2
    N_opt = np.sqrt(compute_budget_flops / 120)
    D_opt = 20 * N_opt
    return N_opt, D_opt


print("=" * 70)
print("시나리오: 다양한 GPU 예산으로 Chinchilla Optimal 모델")
print("=" * 70)
print(f"GPU: A100 (실효 156 TFLOPS), $2/hour")
print()

budgets = [
    ('1 GPU x 1 day',    1 * 24),
    ('1 GPU x 1 week',   1 * 24 * 7),
    ('8 GPUs x 1 week',  8 * 24 * 7),
    ('64 GPUs x 1 month', 64 * 24 * 30),
    ('512 GPUs x 3 months', 512 * 24 * 90),
]

print(f"{'Budget':<25} {'FLOPs':>12} {'Opt Params':>12} {'Opt Tokens':>12} {'Cost':>10}")
print("-" * 75)

for name, gpu_hours in budgets:
    flops = gpu_hours * 156e12 * 3600  # GPU-hours -> FLOPs
    N_opt, D_opt = chinchilla_optimal(flops)
    cost = gpu_hours * 2.0

    # 읽기 쉽게 포맷팅
    def fmt_num(n):
        if n >= 1e12: return f"{n/1e12:.1f}T"
        if n >= 1e9:  return f"{n/1e9:.1f}B"
        if n >= 1e6:  return f"{n/1e6:.1f}M"
        return f"{n:.0f}"

    print(f"{name:<25} {fmt_num(flops):>12} {fmt_num(N_opt):>12} {fmt_num(D_opt):>12} ${cost:>9,.0f}")

print()
print("주의: 실제로는 multi-GPU 효율, 통신 오버헤드 등으로 더 많은 비용 필요")

In [ ]:
# 예산 대비 효과 시각화: 같은 컴퓨팅으로 다른 전략
fig, ax = plt.subplots(figsize=(10, 6))

# 고정 Compute 예산
C_fixed = 1e21  # FLOPs

# 다양한 N 선택지
N_choices = np.logspace(7, 10, 50)
D_choices = C_fixed / (6 * N_choices)

# 각 선택에 대한 Loss 추정 (Chinchilla-style)
# L(N,D) = (N_c/N)^alpha_N + (D_c/D)^alpha_D + L_inf
alpha_N = 0.076
alpha_D = 0.095
N_c = 8.8e13
D_c = 5.4e13
L_inf = 1.69

L_total = (N_c / N_choices) ** alpha_N + (D_c / D_choices) ** alpha_D + L_inf

# Chinchilla optimal
N_opt, D_opt = chinchilla_optimal(C_fixed)
L_opt = (N_c / N_opt) ** alpha_N + (D_c / D_opt) ** alpha_D + L_inf

ax.semilogx(N_choices, L_total, 'b-', linewidth=2, label='Loss(N, D) at fixed C')
ax.axvline(x=N_opt, color='red', linestyle='--', alpha=0.7, label=f'Chinchilla Optimal N={N_opt:.1e}')
ax.scatter([N_opt], [L_opt], color='red', s=100, zorder=5)

ax.set_xlabel('Parameters (N)')
ax.set_ylabel('Estimated Loss')
ax.set_title(f'Fixed Compute C={C_fixed:.0e}: Which Model Size is Best?')
ax.legend()
ax.grid(True, alpha=0.3)

# 주석
ax.annotate('Too small model\n(data wasted)',
            xy=(N_choices[5], L_total[5]),
            xytext=(N_choices[5] * 0.5, L_total[5] + 0.02),
            fontsize=9, ha='center')
ax.annotate('Too large model\n(under-trained)',
            xy=(N_choices[-5], L_total[-5]),
            xytext=(N_choices[-5] * 2, L_total[-5] + 0.02),
            fontsize=9, ha='center')

plt.tight_layout()
plt.show()

print(f"Compute budget: {C_fixed:.0e} FLOPs")
print(f"Chinchilla Optimal: N={N_opt:.2e} params, D={D_opt:.2e} tokens")
print(f"-> 모델이 너무 작으면 데이터가 남고, 너무 크면 데이터가 부족")

### Chinchilla 이후의 트렌드

Chinchilla의 20:1 비율은 **학습 비용**만 최적화한다.

하지만 LLaMA 이후의 트렌드는 **inference 비용**을 고려해서,
작은 모델을 더 많은 데이터로 학습하는 방향으로 바뀌었다.

- 학습 비용: 한 번만 지불
- Inference 비용: 매 요청마다 지불 -> 작은 모델이 유리
- 따라서 LLaMA는 7B 모델에 1T+ 토큰을 학습 (D/N >> 20)

In [ ]:
# 학습 비용 vs Inference 비용 트레이드오프
fig, ax = plt.subplots(figsize=(10, 6))

model_sizes = [1, 3, 7, 13, 30, 65, 175]  # Billion params
training_cost = [s * 20 * 6 / 1e3 for s in model_sizes]  # Chinchilla optimal로 학습시 (arbitrary units)

# Inference 비용: 파람 수에 비례, 1년간 하루 100만 요청 가정
requests_per_year = 365 * 1e6
inference_cost = [s * requests_per_year / 1e12 for s in model_sizes]

total_cost = [t + i for t, i in zip(training_cost, inference_cost)]

ax.plot(model_sizes, training_cost, 'b-o', label='Training Cost (one-time)', linewidth=2)
ax.plot(model_sizes, inference_cost, 'r-o', label='Inference Cost (1 year, 1M req/day)', linewidth=2)
ax.plot(model_sizes, total_cost, 'g--o', label='Total Cost', linewidth=2, alpha=0.7)

ax.set_xlabel('Model Size (Billion Parameters)')
ax.set_ylabel('Relative Cost (arbitrary units)')
ax.set_title('Training vs Inference Cost Trade-off')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print("대규모 서비스에서는 inference 비용 >> training 비용")
print("-> 작은 모델을 더 많이 학습하는 것이 총 비용 최적화에 유리")
print("-> LLaMA, Mistral 등의 전략")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Compute Budget Planner

당신이 A100 GPU 8장을 2주간 사용할 수 있다고 가정하자.

1. 총 Compute 예산(FLOPs)을 계산하세요 (A100 실효 156 TFLOPS)
2. Chinchilla Optimal의 N(파람 수)과 D(토큰 수)를 구하세요
3. 만약 3B 모델을 학습한다면 얼마나 많은 토큰을 사용할 수 있는지 계산하세요
4. 3B 모델이 Chinchilla Optimal이 되려면 몇 토큰이 필요한지 비교하세요

In [ ]:
# TODO: Compute Budget Planner
# 1. 총 GPU hours = 8 GPUs * 14 days * 24 hours
# 2. 총 FLOPs = GPU hours * 156 TFLOPS * 3600 seconds * 1e12
# 3. Chinchilla optimal: N_opt = sqrt(C / 120), D_opt = 20 * N_opt
# 4. 3B 모델: D_available = C / (6 * 3e9)
# 5. 3B Chinchilla optimal: D_needed = 20 * 3e9 = 60B tokens


---
## 핵심 정리

| 개념 | 설명 |
|------|------|
| Power Law | Loss는 N, D, C에 대해 멱법칙 관계 |
| Kaplan (2020) | 모델을 키우는 것이 더 효과적 |
| Chinchilla (2022) | D = 20N이 최적, 모델과 데이터 균형 중요 |
| Compute Budget | C = 6ND, A100 기준 비용 계산 가능 |
| Inference 최적화 | LLaMA 이후: 작은 모델 + 많은 데이터 (D/N >> 20) |
| L_inf | 줄일 수 없는 손실 (entropy of natural text) |

**다음 노트북**: [03-training-techniques.ipynb](03-training-techniques.ipynb) - 대규모 학습 기법